In [ ]:
%%sql

-- ============================================================
-- GOLD 6 - EVACUATION ROUTES
-- ============================================================
--
-- IMPORTANTE:
--   Índice de idoneidad de carreteras próximas
--   al incendio para análisis de evacuación potencial.
--
-- ============================================================


DROP TABLE IF EXISTS gold_evacuation_routes;


CREATE TABLE gold_evacuation_routes (

    fire_detection_id STRING,
    cluster_id STRING,

    fire_detection_timestamp TIMESTAMP,

    fire_latitude DOUBLE,
    fire_longitude DOUBLE,

    fire_radiative_power DOUBLE,

    osm_way_id STRING,

    road_reference STRING,
    road_name STRING,
    road_classification STRING,

    road_latitude DOUBLE,
    road_longitude DOUBLE,

    distance_to_fire_km DOUBLE,

    max_speed_kmh INT,
    lanes_count INT,

    is_oneway STRING,
    pavement_surface STRING,

    has_bridge STRING,
    has_tunnel STRING,

    active_traffic_incident_count BIGINT,
    traffic_severity STRING,

    distance_score INT,
    capacity_score INT,
    traffic_score INT,
    road_type_score INT,

    evacuation_suitability_score INT,

    evacuation_category STRING,

    source_fire STRING,
    source_osm STRING,

    updated_at TIMESTAMP
);


-- ============================================================
-- 1. NASA
-- ============================================================

CREATE OR REPLACE TEMP VIEW evacuation_fires AS

SELECT

    SHA2(
        CONCAT_WS(
            '|',
            CAST(latitude AS STRING),
            CAST(longitude AS STRING),
            CAST(fire_detection_timestamp AS STRING)
        ),
        256
    ) AS fire_detection_id,

    cluster_id,

    fire_detection_timestamp,

    CAST(latitude AS DOUBLE) AS fire_latitude,
    CAST(longitude AS DOUBLE) AS fire_longitude,

    CAST(fire_radiative_power AS DOUBLE)
        AS fire_radiative_power,

    landing_source_file AS source_fire

FROM silver_nasa_fires

WHERE latitude IS NOT NULL
  AND longitude IS NOT NULL
  AND fire_detection_timestamp IS NOT NULL;


-- ============================================================
-- 2. OSM
-- ============================================================

CREATE OR REPLACE TEMP VIEW evacuation_roads AS

SELECT

    CAST(osm_way_id AS STRING) AS osm_way_id,

    road_reference,
    road_name,
    road_classification,

    CAST(centroid_latitude AS DOUBLE)
        AS road_latitude,

    CAST(centroid_longitude AS DOUBLE)
        AS road_longitude,

    CAST(max_speed_kmh AS INT)
        AS max_speed_kmh,

    CAST(lanes_count AS INT)
        AS lanes_count,

    is_oneway,
    pavement_surface,

    has_bridge,
    has_tunnel,

    landing_source_file AS source_osm

FROM silver_osm_roads

WHERE osm_way_id IS NOT NULL
  AND centroid_latitude IS NOT NULL
  AND centroid_longitude IS NOT NULL;


-- ============================================================
-- 3. Distancia
-- ============================================================

CREATE OR REPLACE TEMP VIEW evacuation_candidates AS

SELECT

    f.fire_detection_id,
    f.cluster_id,

    f.fire_detection_timestamp,

    f.fire_latitude,
    f.fire_longitude,

    f.fire_radiative_power,

    r.osm_way_id,

    r.road_reference,
    r.road_name,
    r.road_classification,

    r.road_latitude,
    r.road_longitude,

    r.max_speed_kmh,
    r.lanes_count,

    r.is_oneway,
    r.pavement_surface,

    r.has_bridge,
    r.has_tunnel,

    r.source_osm,
    f.source_fire,

    (
        6371.0 * 2.0 * ASIN(
            SQRT(

                POWER(
                    SIN(
                        RADIANS(
                            r.road_latitude
                            - f.fire_latitude
                        ) / 2.0
                    ),
                    2
                )

                +

                COS(
                    RADIANS(f.fire_latitude)
                )
                *
                COS(
                    RADIANS(r.road_latitude)
                )
                *
                POWER(
                    SIN(
                        RADIANS(
                            r.road_longitude
                            - f.fire_longitude
                        ) / 2.0
                    ),
                    2
                )

            )
        )
    ) AS distance_to_fire_km

FROM evacuation_fires f

INNER JOIN evacuation_roads r

    ON r.road_latitude BETWEEN
        f.fire_latitude - 0.25
        AND
        f.fire_latitude + 0.25

   AND r.road_longitude BETWEEN
        f.fire_longitude - 0.35
        AND
        f.fire_longitude + 0.35;


-- ============================================================
-- 4. Tráfico activo
-- ============================================================

CREATE OR REPLACE TEMP VIEW evacuation_traffic AS

SELECT

    e.fire_detection_id,
    e.osm_way_id,

    COUNT(
        DISTINCT d.record_id
    ) AS active_traffic_incident_count,

    CASE

        WHEN MAX(
            CASE

                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) IN ('CRITICAL', 'SEVERE')
                    THEN 4

                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) = 'HIGH'
                    THEN 3

                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) = 'MEDIUM'
                    THEN 2

                ELSE 1

            END
        ) = 4
            THEN 'CRITICAL'

        WHEN MAX(
            CASE

                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) = 'HIGH'
                    THEN 3

                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) = 'MEDIUM'
                    THEN 2

                ELSE 1

            END
        ) = 3
            THEN 'HIGH'

        WHEN MAX(
            CASE

                WHEN UPPER(
                    COALESCE(
                        d.severity_level,
                        'NORMAL'
                    )
                ) = 'MEDIUM'
                    THEN 2

                ELSE 1

            END
        ) = 2
            THEN 'MEDIUM'

        ELSE 'NORMAL'

    END AS traffic_severity

FROM evacuation_candidates e

LEFT JOIN silver_dgt_traffic d

    ON d.latitude IS NOT NULL
   AND d.longitude IS NOT NULL

   AND d.latitude BETWEEN
        e.road_latitude - 0.05
        AND
        e.road_latitude + 0.05

   AND d.longitude BETWEEN
        e.road_longitude - 0.07
        AND
        e.road_longitude + 0.07

   AND d.start_timestamp
        <= e.fire_detection_timestamp

   AND (
        d.end_timestamp IS NULL
        OR
        d.end_timestamp
            >= e.fire_detection_timestamp
   )

GROUP BY

    e.fire_detection_id,
    e.osm_way_id;


-- ============================================================
-- 5. Scoring (Inclusión explícita de source_fire y source_osm)
-- ============================================================

CREATE OR REPLACE TEMP VIEW evacuation_scored AS

SELECT

    e.fire_detection_id,
    e.cluster_id,

    e.fire_detection_timestamp,

    e.fire_latitude,
    e.fire_longitude,

    e.fire_radiative_power,

    e.osm_way_id,

    e.road_reference,
    e.road_name,
    e.road_classification,

    e.road_latitude,
    e.road_longitude,

    e.distance_to_fire_km,

    e.max_speed_kmh,
    e.lanes_count,

    e.is_oneway,
    e.pavement_surface,

    e.has_bridge,
    e.has_tunnel,

    e.source_fire,
    e.source_osm,

    COALESCE(
        t.active_traffic_incident_count,
        0
    ) AS active_traffic_incident_count,

    COALESCE(
        t.traffic_severity,
        'NORMAL'
    ) AS traffic_severity,

    -- DISTANCIA
    CASE
        WHEN e.distance_to_fire_km >= 15 THEN 5
        WHEN e.distance_to_fire_km >= 10 THEN 4
        WHEN e.distance_to_fire_km >= 5  THEN 3
        WHEN e.distance_to_fire_km >= 2  THEN 2
        ELSE 0
    END AS distance_score,

    -- CAPACIDAD
    CASE
        WHEN COALESCE(e.lanes_count, 0) >= 3 THEN 5
        WHEN COALESCE(e.lanes_count, 0) = 2  THEN 4
        WHEN COALESCE(e.lanes_count, 0) = 1  THEN 2
        ELSE 1
    END AS capacity_score,

    -- TRÁFICO
    CASE
        WHEN COALESCE(t.active_traffic_incident_count, 0) = 0 THEN 5
        WHEN UPPER(COALESCE(t.traffic_severity, 'NORMAL')) IN ('CRITICAL', 'SEVERE') THEN 0
        WHEN UPPER(COALESCE(t.traffic_severity, 'NORMAL')) = 'HIGH' THEN 1
        WHEN COALESCE(t.active_traffic_incident_count, 0) = 1 THEN 3
        ELSE 1
    END AS traffic_score,

    -- TIPO DE CARRETERA
    CASE
        WHEN LOWER(COALESCE(e.road_classification, '')) = 'motorway' THEN 5
        WHEN LOWER(COALESCE(e.road_classification, '')) = 'trunk' THEN 4
        WHEN LOWER(COALESCE(e.road_classification, '')) IN ('primary', 'secondary') THEN 3
        ELSE 2
    END AS road_type_score

FROM evacuation_candidates e

LEFT JOIN evacuation_traffic t

    ON e.fire_detection_id =
       t.fire_detection_id

   AND e.osm_way_id =
       t.osm_way_id

WHERE e.distance_to_fire_km <= 25.0;


-- ============================================================
-- 6. Score final
-- ============================================================

CREATE OR REPLACE TEMP VIEW gold_evacuation_source AS

SELECT

    fire_detection_id,
    cluster_id,

    fire_detection_timestamp,

    fire_latitude,
    fire_longitude,

    fire_radiative_power,

    osm_way_id,

    road_reference,
    road_name,
    road_classification,

    road_latitude,
    road_longitude,

    distance_to_fire_km,

    max_speed_kmh,
    lanes_count,

    is_oneway,
    pavement_surface,

    has_bridge,
    has_tunnel,

    active_traffic_incident_count,
    traffic_severity,

    distance_score,
    capacity_score,
    traffic_score,
    road_type_score,

    (
        distance_score
        + capacity_score
        + traffic_score
        + road_type_score
    ) AS evacuation_suitability_score,

    CASE

        WHEN (
            distance_score
            + capacity_score
            + traffic_score
            + road_type_score
        ) >= 16

            THEN 'PREFERRED'

        WHEN (
            distance_score
            + capacity_score
            + traffic_score
            + road_type_score
        ) >= 12

            THEN 'MONITOR'

        WHEN (
            distance_score
            + capacity_score
            + traffic_score
            + road_type_score
        ) >= 8

            THEN 'CAUTION'

        ELSE 'AVOID'

    END AS evacuation_category,

    source_fire,
    source_osm,

    current_timestamp() AS updated_at

FROM evacuation_scored;


-- ============================================================
-- 7. MERGE
-- ============================================================

MERGE INTO gold_evacuation_routes AS target

USING gold_evacuation_source AS source

ON target.fire_detection_id =
       source.fire_detection_id

AND target.osm_way_id =
       source.osm_way_id

WHEN MATCHED THEN UPDATE SET

    target.cluster_id =
        source.cluster_id,

    target.fire_detection_timestamp =
        source.fire_detection_timestamp,

    target.fire_latitude =
        source.fire_latitude,

    target.fire_longitude =
        source.fire_longitude,

    target.fire_radiative_power =
        source.fire_radiative_power,

    target.road_reference =
        source.road_reference,

    target.road_name =
        source.road_name,

    target.road_classification =
        source.road_classification,

    target.road_latitude =
        source.road_latitude,

    target.road_longitude =
        source.road_longitude,

    target.distance_to_fire_km =
        source.distance_to_fire_km,

    target.max_speed_kmh =
        source.max_speed_kmh,

    target.lanes_count =
        source.lanes_count,

    target.is_oneway =
        source.is_oneway,

    target.pavement_surface =
        source.pavement_surface,

    target.has_bridge =
        source.has_bridge,

    target.has_tunnel =
        source.has_tunnel,

    target.active_traffic_incident_count =
        source.active_traffic_incident_count,

    target.traffic_severity =
        source.traffic_severity,

    target.distance_score =
        source.distance_score,

    target.capacity_score =
        source.capacity_score,

    target.traffic_score =
        source.traffic_score,

    target.road_type_score =
        source.road_type_score,

    target.evacuation_suitability_score =
        source.evacuation_suitability_score,

    target.evacuation_category =
        source.evacuation_category,

    target.source_fire =
        source.source_fire,

    target.source_osm =
        source.source_osm,

    target.updated_at =
        source.updated_at


WHEN NOT MATCHED THEN INSERT (

    fire_detection_id,
    cluster_id,

    fire_detection_timestamp,

    fire_latitude,
    fire_longitude,

    fire_radiative_power,

    osm_way_id,

    road_reference,
    road_name,
    road_classification,

    road_latitude,
    road_longitude,

    distance_to_fire_km,

    max_speed_kmh,
    lanes_count,

    is_oneway,
    pavement_surface,

    has_bridge,
    has_tunnel,

    active_traffic_incident_count,
    traffic_severity,

    distance_score,
    capacity_score,
    traffic_score,
    road_type_score,

    evacuation_suitability_score,
    evacuation_category,

    source_fire,
    source_osm,

    updated_at

)

VALUES (

    source.fire_detection_id,
    source.cluster_id,

    source.fire_detection_timestamp,

    source.fire_latitude,
    source.fire_longitude,

    source.fire_radiative_power,

    source.osm_way_id,

    source.road_reference,
    source.road_name,
    source.road_classification,

    source.road_latitude,
    source.road_longitude,

    source.distance_to_fire_km,

    source.max_speed_kmh,
    source.lanes_count,

    source.is_oneway,
    source.pavement_surface,

    source.has_bridge,
    source.has_tunnel,

    source.active_traffic_incident_count,
    source.traffic_severity,

    source.distance_score,
    source.capacity_score,
    source.traffic_score,
    source.road_type_score,

    source.evacuation_suitability_score,
    source.evacuation_category,

    source.source_fire,
    source.source_osm,

    source.updated_at

);